# Nova — Smart Home Assistant

One-click entry point. Run cells top-to-bottom.


In [1]:
# Cell 1: Install dependencies (first run only)
import sys
!{sys.executable} -m pip install -q numpy pandas sounddevice soundfile faster-whisper \
    transformers accelerate sentencepiece pyttsx3 chromadb sentence-transformers bitsandbytes


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip3 install --upgrade pip


In [2]:
# Cell 2: Add project root to sys.path + HF authentication
import sys, os

PROJECT_ROOT = os.path.dirname(os.path.abspath("main.ipynb"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

# Set HF_TOKEN in terminal before launching Jupyter:
#   export HF_TOKEN=hf_...
from huggingface_hub import login
token = os.environ.get("HF_TOKEN", "")
if token:
    login(token=token)
    print("HF login done.")
else:
    print("HF_TOKEN not set — public models still work.")

Project root: /Users/yiwenchen/Desktop/CSEE6895/final_project/6895_Final_project
HF_TOKEN not set — public models still work.


In [3]:
# Cell 3: Load LLM parser
from llm_parser import LLMParser

llm = LLMParser()

Loading LLM (Qwen/Qwen2.5-3B-Instruct) on mps [torch.float16] ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

LLM ready.


In [4]:
# Cell 4: Load STT + TTS
from audio import STTModel, TTSEngine

stt = STTModel()
tts = TTSEngine()

Loading STT (tiny.en) ...
STT ready.


In [5]:
# Cell 5: Load embedding model + initialize memory
from sentence_transformers import SentenceTransformer
from memory import MemoryManager

_embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embed = lambda text: _embed_model.encode(text, convert_to_numpy=True).tolist()

memory = MemoryManager(embed_fn=embed)
print("Memory summary:", memory.summary())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Memory summary: {'working_turns': 0, 'episodic_count': 7, 'prefs': {}, 'skills_count': 0}


In [7]:
# Cell 7: Text demo — direct command, clarification, invalid filter, memory QA

SEP = "=" * 62

def run_test(label, text, agent, reset=True):
    if reset:
        agent.reset_dialogue()
    print("\n" + SEP)
    print(f"[{label}]")
    print(f"  input : {text!r}")
    result = agent.handle(text, verbose=True)
    t     = result['semantic'].get('type', '?')
    reply = result.get('spoken_reply', '—')
    ms    = result.get('llm_latency_ms', '?')
    exec_ = result.get('execution', '—')
    print(f"  type  : {t}")
    print(f"  reply : {reply!r}")
    print(f"  exec  : {exec_}  |  latency: {ms} ms")
    return result

# ── 1. Direct command ──────────────────────────────────────────────────────────
run_test("direct_command", "Nova, turn on the light.", nova)

# ── 2. Clarification ───────────────────────────────────────────────────────────
run_test("needs_clarification", "Nova, I feel a bit cold.", nova)

# ── 3. Invalid (no wake word) ──────────────────────────────────────────────────
run_test("no_wakeword", "How are you today?", nova)

# ── 4. Memory QA: tell name → recall name ─────────────────────────────────────
# Two turns WITHOUT reset_dialogue() so working memory persists across them.
print("\n" + SEP)
print("[memory_test] multi-turn: user introduces name, then asks Nova to recall it")
nova.reset_dialogue()

run_test("tell_name",   "Nova, my name is Alice.",  nova, reset=False)
run_test("recall_name", "Nova, what is my name?",   nova, reset=False)


SyntaxError: unterminated f-string literal (detected at line 30) (1394599363.py, line 30)

In [7]:
# Cell 7: Quick text demo
demo_inputs = [
    "Nova, turn on the light.",
    "Nova, I feel a bit cold.",
    "Nova, how do you feel today",
    "How are you today?",   # no Nova prefix → should be filtered
]

for text in demo_inputs:
    nova.reset_dialogue()
    print("\n" + "=" * 60)
    result = nova.handle(text, verbose=True)
    print(f"→ type={result['semantic'].get('type')}  execution={result.get('execution')}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



STT text: Nova, turn on the light.
Raw output: {"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Latency: 19530.4 ms
[TTS] Sorry, I didn't understand that.
→ type=invalid  execution=SKIPPED

STT text: Nova, I feel a bit cold.
Raw output: {"type":"needs_clarification","question":"Would you like me to close the window or raise the AC temperature?","options":["close_window","raise_ac_temperature"],"reply":"Would you like me to close the window or raise the AC temperature?"}
Latency: 12055.9 ms
[Clarification options]
  [1] Close the window
  [2] Raise the AC temperature
[TTS] Would you like me to close the window or raise the AC temperature? Here are your options: Option 1: Close the window. Option 2: Raise the AC temperature. Please say the option number or describe what you want.
→ type=needs_clarification  execution=PENDING_USER_REPLY

STT text: Nova, how do you feel today
R

In [ ]:
# Cell 8: Start continuous audio loop (microphone required)
# Uncomment to run; press Ctrl+C to stop.

# from audio import AudioListener
# listener = AudioListener(agent=nova, stt=stt)
# listener.continuous_loop()